# Step 1: Preprocessing Raw IPL Dataset

This notebook covers:
1. Loading the raw `deliveries.csv` and `matches.csv` files.
2. Merging matches data into delivery details to get match contexts.
3. Restructuring features specifically for chasing (second innings) dynamics.
4. Outputting a processed dataset for training win probability predictors.

In [ ]:
import pandas as pd
import numpy as np
import os

## 1. Load CSV Files

In [ ]:
deliveries = pd.read_csv("../dataset/raw/deliveries.csv")
matches = pd.read_csv("../dataset/raw/matches.csv")
print("Deliveries shape:", deliveries.shape)
print("Matches shape:", matches.shape)

## 2. Basic Cleaning and Formatting

In [ ]:
# Remove duplicates
deliveries = deliveries.drop_duplicates()
matches = matches.drop_duplicates()

# Handle boolean values in Wickets
deliveries["is_wicket"] = deliveries["is_wicket"].astype(str).str.upper().map({
    "TRUE": 1,
    "FALSE": 0
}).fillna(0).astype(int)

# Clean matches to exclude abandoned/no-result games
matches_clean = matches[(matches["result"] == "normal") & (matches["winner"].notna())].copy()
matches_clean["target_score"] = matches_clean["first_innings_score"] + 1

## 3. Chasing Innings Preprocessing

In [ ]:
# Extract second innings only
second_innings = deliveries[deliveries["innings"] == 2].copy()

# Merge target score and winner details
data = second_innings.merge(
    matches_clean[["match_id", "winner", "target_score"]],
    on="match_id",
    how="inner"
)

# Sort balls
data = data.sort_values(["match_id", "over", "ball", "delivery_id"])

# Distinguish legal balls (no-balls & wides do not reduce remaining balls)
data["extra_type"] = data["extra_type"].fillna("").astype(str).str.lower().str.strip()
data["legal_ball"] = (~data["extra_type"].isin(["wide", "wides", "no ball", "noball", "no-ball"])).astype(int)

## 4. Run-Time Metrics Calculations

In [ ]:
# Cumulative score and wickets lost
data["current_score"] = data.groupby("match_id")["total_runs"].cumsum()
data["wickets_lost"] = data.groupby("match_id")["is_wicket"].cumsum()
data["balls_bowled"] = data.groupby("match_id")["legal_ball"].cumsum()

# Dynamic metrics
data["balls_left"] = 120 - data["balls_bowled"]
data["runs_left"] = data["target_score"] - data["current_score"]
data["wickets_left"] = 10 - data["wickets_lost"]
data["chasing_team"] = data["batting_team"]

# Chasing team won? (1 = yes, 0 = no)
data["chasing_team_won"] = (data["chasing_team"] == data["winner"]).astype(int)

# Run Rates
data["current_run_rate"] = np.where(data["balls_bowled"] > 0, data["current_score"] / (data["balls_bowled"] / 6.0), 0.0)
data["required_run_rate"] = np.where(data["balls_left"] > 0, data["runs_left"] / (data["balls_left"] / 6.0), 0.0)

## 5. Filter Invalid Rows and Export Features

In [ ]:
final_data = data[
    (data["balls_left"] >= 0) &
    (data["balls_left"] <= 120) &
    (data["wickets_left"] >= 0) &
    (data["wickets_left"] <= 10) &
    (data["runs_left"] >= 0)
].copy()

features = [
    "target_score",
    "current_score",
    "runs_left",
    "balls_left",
    "wickets_left",
    "current_run_rate",
    "required_run_rate"
]
target = "chasing_team_won"
metadata = ["match_id", "chasing_team", "bowling_team"]

ml_dataset = final_data[features + [target] + metadata]

os.makedirs("../dataset/processed", exist_ok=True)
ml_dataset.to_csv("../dataset/processed/win_probability_dataset.csv", index=False)
print("Saved preprocessed dataset. Final shape:", ml_dataset.shape)